## Step 1: Rust Environment Setup
Google Colab instances run on Ubuntu but do not have Rust installed by default. This cell performs the following actions:
* Downloads and executes the `rustup` installer script directly from the official Rust website.
* Uses Python's `os` module to append the newly created `.cargo/bin` directory to the system's `PATH`. This ensures that commands like `rustc` (the compiler) and `cargo` (the package manager) are recognized by the Colab terminal.
* Verifies the installation by printing the installed Rust compiler version.

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ":/root/.cargo/bin"
!rustc --version

info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-07-16 for version 1.97.1 (8bab26f4f 2026-07-14)
info: downloading 6 components
        cargo downloading [               ]         0 B (0 B/s, ETA: 0s)
        cargo downloading [               ]   10.63 MiB (0 B/s, ETA: 0s)
        cargo downloading [#              ]   10.63 MiB (3.79 MiB/s, ETA: 3s)
        cargo downloading [#              ]   10.63 MiB (3.79 MiB/s, ETA: 3s)
        cargo downloading [#              ]   10.63 MiB (3.79 MiB/s, ETA: 3s)
        cargo downloading [#              ]   10.63 MiB (3.79 MiB/s, ETA: 3s)
        car

## Step 2: Initialize Cargo Project
Rust uses a powerful package manager and build system called **Cargo**.
* `!cargo new agent_swarm --bin` tells Cargo to create a new directory named `agent_swarm` containing the boilerplate for a binary (executable) application.
* `%cd agent_swarm` is a Jupyter magic command that changes the notebook's current working directory to our new project folder, ensuring all subsequent commands run in the correct context.

In [ ]:
!cargo new agent_swarm --bin
%cd agent_swarm

    Creating binary (application) `agent_swarm` package
note: see more `Cargo.toml` keys and their definitions at https://doc.rust-lang.org/cargo/reference/manifest.html
/content/agent_swarm


## Step 3: The AI Swarm Architecture
This cell uses the `%%writefile` magic command to overwrite the default `src/main.rs` file with our custom swarm logic. Here is a breakdown of the architecture:

1. **Core Data Structures:**
   * `AgentRole`: An enum defining the three distinct agents (Researcher, Synthesizer, Critic).
   * `Message`: A struct representing a piece of communication, containing the sender's role and the text content.
   * `SwarmContext`: This acts as a "Blackboard" or shared memory. It holds the overarching `goal` and a history of all messages passed between agents.

2. **The `Agent` Trait:**
   * A shared interface that all agents must implement. It requires a `role()` function and a `process()` function that takes mutable access to the `SwarmContext`.

3. **Agent Implementations:**
   * **ResearcherAgent:** Gathers initial context and mock data based on the goal.
   * **SynthesizerAgent:** Reads the history, finds the Researcher's last message, and drafts a synthesized response.
   * **CriticAgent:** Evaluates the Synthesizer's draft and provides final validation.

4. **The Execution Loop (`main`):**
   * Initializes the shared context and the agents.
   * Runs a synchronous multi-turn loop where agents take turns processing data and broadcasting their thoughts back to the shared context.

In [ ]:
%%writefile src/main.rs
use std::collections::VecDeque;

// Define the roles in our AI agent swarm
#[derive(Debug, Clone)]
enum AgentRole {
    Researcher,
    Synthesizer,
    Critic,
}

// Define the message structure passed between agents
#[derive(Debug, Clone)]
struct Message {
    sender: AgentRole,
    content: String,
}

// Define the shared state/blackboard of the swarm
struct SwarmContext {
    goal: String,
    memory: VecDeque<Message>,
}

impl SwarmContext {
    fn new(goal: &str) -> Self {
        Self {
            goal: goal.to_string(),
            memory: VecDeque::new(),
        }
    }

    fn broadcast(&mut self, sender: AgentRole, content: &str) {
        println!("\n[{:?}]: {}", sender, content);
        self.memory.push_back(Message {
            sender,
            content: content.to_string(),
        });
    }
}

// Trait representing an autonomous swarm agent
trait Agent {
    fn role(&self) -> AgentRole;
    fn process(&self, context: &mut SwarmContext);
}

// 1. Researcher Agent: Gathers information/context
struct ResearcherAgent;
impl Agent for ResearcherAgent {
    fn role(&self) -> AgentRole { AgentRole::Researcher }
    fn process(&self, context: &mut SwarmContext) {
        let thought = format!(
            "Analyzing parameters for objective: '{}'. Retrieved mock semantic embeddings and knowledge chunks.",
            context.goal
        );
        context.broadcast(self.role(), &thought);
    }
}

// 2. Synthesizer Agent: Combines research into a coherent draft
struct SynthesizerAgent;
impl Agent for SynthesizerAgent {
    fn role(&self) -> AgentRole { AgentRole::Synthesizer }
    fn process(&self, context: &mut SwarmContext) {
        let last_research = context.memory.iter()
            .rfind(|m| matches!(m.sender, AgentRole::Researcher))
            .map(|m| m.content.as_str())
            .unwrap_or("No data");

        let thought = format!(
            "Synthesizing structured output based on research findings: \"{}\"",
            last_research
        );
        context.broadcast(self.role(), &thought);
    }
}

// 3. Critic Agent: Evaluates the synthesized output for safety and accuracy
struct CriticAgent;
impl Agent for CriticAgent {
    fn role(&self) -> AgentRole { AgentRole::Critic }
    fn process(&self, context: &mut SwarmContext) {
        let last_synthesis = context.memory.iter()
            .rfind(|m| matches!(m.sender, AgentRole::Synthesizer))
            .map(|m| m.content.as_str())
            .unwrap_or("No draft");

        let thought = format!(
            "Critiquing draft (\"{}\"). Validation check passed: Swarm consensus reached successfully!",
            last_synthesis
        );
        context.broadcast(self.role(), &thought);
    }
}

fn main() {
    println!("=== Initializing Rust AI Agent Swarm ===");

    // Initialize shared state with a target goal
    let mut context = SwarmContext::new("Optimize decentralized vector database retrieval latency");

    // Instantiate agents
    let researcher = ResearcherAgent;
    let synthesizer = SynthesizerAgent;
    let critic = CriticAgent;

    // Execute multi-turn swarm loop (cooperative execution)
    let max_iterations = 2;
    for turn in 1..=max_iterations {
        println!("\n--- Swarm Execution Iteration {} ---", turn);
        researcher.process(&mut context);
        synthesizer.process(&mut context);
        critic.process(&mut context);
    }

    println!("\n=== Swarm Task Completed Successfully! ===");
}

Overwriting src/main.rs


## Step 4: Compile and Execute
With our source code written, we use Cargo to compile and run the project.
* The `--release` flag instructs the Rust compiler to apply aggressive optimizations. While it takes slightly longer to compile, the resulting binary runs significantly faster, which is standard practice for performance-critical AI workloads in Rust.

In [ ]:
!cargo run --release

   Compiling agent_swarm v0.1.0 (/content/agent_swarm)
    Finished `release` profile [optimized] target(s) in 0.22s
     Running `target/release/agent_swarm`
=== Initializing Rust AI Agent Swarm ===

--- Swarm Execution Iteration 1 ---

[Researcher]: Analyzing parameters for objective: 'Optimize decentralized vector database retrieval latency'. Retrieved mock semantic embeddings and knowledge chunks.

[Synthesizer]: Synthesizing structured output based on research findings: "Analyzing parameters for objective: 'Optimize decentralized vector database retrieval latency'. Retrieved mock semantic embeddings and knowledge chunks."

[Critic]: Critiquing draft ("Synthesizing structured output based on research findings: "Analyzing parameters for objective: 'Optimize decentralized vector database retrieval latency'. Retrieved mock semantic embeddings and knowledge chunks.""). Validation check passed: Swarm consensus reached successfully!

--- Swarm Execution Iteration 2 ---

[Researcher]: Ana

Based on the exact code and instructions I provided earlier, here is how both of those apply:

### 1. Name of the Notebook / Project

I did not give a specific `.ipynb` filename for the Google Colab notebook itself (it defaults to `Untitled0.ipynb` unless you rename it). However, the **name of the Rust project** we created inside the notebook is **`agent_swarm`**.

This was defined in the setup cell:

```bash
!cargo new agent_swarm --bin

```

### 2. The IPO (Input-Process-Output) Model in the Code

In software engineering, **IPO** stands for **Input, Process, and Output**. The Rust code I gave you perfectly mirrors this concept within the `Agent` trait and how it interacts with the shared `SwarmContext`.

Here is exactly how IPO is mapped in the provided code:

* **Input (I):** The input is the `SwarmContext` (the shared blackboard) passed to each agent. It contains the overarching `goal` (e.g., *"Optimize decentralized vector database retrieval latency"*) and the `memory` of previous messages.
```rust
fn process(&self, context: &mut SwarmContext) // 'context' is the Input

```


* **Process (P):** This is the internal logic inside each agent's `process()` block. For example, the **SynthesizerAgent** takes the input, searches the memory for the Researcher's data, and processes it into a new draft.
```rust
let last_research = context.memory.iter()
    .rfind(|m| matches!(m.sender, AgentRole::Researcher))
    .map(|m| m.content.as_str())
    .unwrap_or("No data");

```


* **Output (O):** The agent outputs its result by calling the `broadcast` method, which logs the thought to the console and pushes a new `Message` back into the shared `context.memory` for the next agent to use.
```rust
context.broadcast(self.role(), &thought); // This generates the Output

```